# F01 Transformer 形状与注意力读图


## 这个基础 lab 是为了解决什么

这本 lab 用来消除最常见的小白阻塞项，再进入文章优先主线。


## 做完后你应该带走的技能

- 解释 token、embedding、attention score 和 context vector 的关系。
- 手动读一张 attention heatmap，而不是只说“模型在看这里”。
- 把 residual stream 看成逐步累加的计算缓存。


## 最后交付这些产物

- 1 张手工标注过的 attention heatmap。
- 1 段用自然语言解释的 residual update。
- 1 份把张量形状写清楚的 notebook 注释。


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Jonny-English/learn-interpretability.git"
REPO_DIR = "learn-interpretability"
REPO_BRANCH = "main"

if "google.colab" in sys.modules:
    candidate = Path("/content") / REPO_DIR
    if not candidate.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(candidate)],
            check=True,
        )
    os.chdir(candidate)

root = Path.cwd().resolve()
while not (root / "content" / "course.json").exists():
    if root.parent == root:
        raise RuntimeError("Run this notebook from the repository root.")
    root = root.parent

import math
import matplotlib.pyplot as plt
import numpy as np

tokens = ["Ada", "wrote", "the", "patch", "carefully"]
embeddings = np.array([
    [1.0, 0.1, 0.3],
    [0.8, 0.4, 0.2],
    [0.2, 0.9, 0.1],
    [0.7, 0.3, 0.9],
    [0.9, 0.2, 0.8],
])
Wq = np.array([[1.0, 0.2], [0.1, 0.9], [0.7, 0.3]])
Wk = np.array([[0.9, 0.1], [0.2, 1.0], [0.6, 0.4]])
Wv = np.array([[0.4, 0.8], [0.9, 0.3], [0.3, 0.7]])

Q = embeddings @ Wq
K = embeddings @ Wk
V = embeddings @ Wv
scores = Q @ K.T / math.sqrt(K.shape[-1])
scores = scores - scores.max(axis=-1, keepdims=True)
weights = np.exp(scores)
weights = weights / weights.sum(axis=-1, keepdims=True)
contexts = weights @ V
residual_after = np.concatenate([embeddings[:, :2] + contexts, embeddings[:, 2:]], axis=-1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
im = axes[0].imshow(weights, cmap="Blues")
axes[0].set_xticks(range(len(tokens)), tokens, rotation=30)
axes[0].set_yticks(range(len(tokens)), tokens)
axes[0].set_title("Attention weights")
plt.colorbar(im, ax=axes[0], fraction=0.046)

axes[1].plot(residual_after[-1], marker="o", color="#c96a28")
axes[1].set_title("Final-token residual update")
axes[1].set_xlabel("channel")
plt.tight_layout()

print("Q shape:", Q.shape, "| K shape:", K.shape, "| V shape:", V.shape)
print("Most attended token for the final position:", tokens[int(weights[-1].argmax())])
print("Final-position context vector:", np.round(contexts[-1], 3))


## 小结

在你能读 circuit 之前，先要能不糊弄地读懂形状、attention 矩阵和 residual update。


## 验收题

- attention 权重高，并不自动等于“因果贡献大”，为什么？
- residual stream 为什么比“某个单一神经元”更适合作为后续 tracing 的语言？
- 如果你看不懂 Q、K、V 的形状，后面的 circuit 解释会卡在哪里？
- 如果你不能在不重开 notebook 的情况下独立答出至少 2 题，就先不要进入文章主线。
